# DART Report Reader — Colab 실행 가이드

DART OpenAPI 인증키 발급: https://opendart.fss.or.kr/uat/uia/egovLoginUsr.do

**Colab Secrets 설정 방법**  
왼쪽 사이드바 🔑 아이콘 → `DART_API_KEY` 이름으로 API 키 등록


## 1. 설치

In [ ]:
!pip install git+https://github.com/syuririk/DR.git -q
# 로컬 개발 중일 때
# !pip install -e /content/DR -q

## 2. API 키 설정

In [ ]:
import os

# 방법 A: Colab Secrets (권장) — 왼쪽 🔑 아이콘에서 DART_API_KEY 등록
from google.colab import userdata
os.environ['DART_API_KEY'] = userdata.get('DART_API_KEY')

# 방법 B: 직접 입력 (Secrets 사용 불가 시)
# os.environ['DART_API_KEY'] = 'YOUR_DART_API_KEY_HERE'

print('API 키 설정 완료')

## 3. 설정 & 초기화

In [ ]:
from dart_report_reader import DartReportReader, DartConfig, ReportCode, SectionCode

cfg = DartConfig(
    api_key=os.environ['DART_API_KEY'],
    cache_dir='/content/dart_cache',
    output_dir='/content/dart_vault',
    timeout=30,
    retry=3,
)
print(f'캐시 경로: {cfg.cache_dir}')
print(f'출력 경로: {cfg.output_dir}')

In [ ]:
reader = DartReportReader(cfg)
reader.init()
print('초기화 완료!')

## 4. 기업 검색

In [ ]:
results = reader.search_corp('삼성')
for r in results[:5]:
    print(f"{r['corp_name']:20s}  종목코드: {r['stock_code']:6s}  고유번호: {r['corp_code']}")

## 5. 단일 보고서 파싱 (DS002 정형 데이터)

In [ ]:
report = reader.fetch_report(
    corp='삼성전자',
    bsns_year=2023,
    reprt_code=ReportCode.ANNUAL,
    sections=[SectionCode.DIVIDEND, SectionCode.EXECUTIVE, SectionCode.EMPLOYEE],
)
print(f'회사명: {report.corp_name}')
print(f'파싱 섹션: {[SectionCode.label(s) for s in report.available_sections]}')

## 6. 원문 파싱 (DS001 HTML 뷰어 방식)

In [ ]:
# 접수번호를 알고 있을 때 — 특정 섹션만 로딩
doc = reader.fetch_document(
    rcept_no='20240312000736',       # 삼성전자 2023 사업보고서
    section_filter=['기타 참고사항'],  # 이 키워드를 포함한 섹션만 HTTP 요청
)
print(f'기업: {doc.corp_name}  /  섹션 수: {len(doc.all_sections_flat())}')
target = doc.find_section(['기타 참고사항'])
if target:
    print(f'\n[{target.title}]')
    print(target.content_md[:500])

## 7. Obsidian Vault 생성

In [ ]:
paths = reader.build_vault(
    corp='삼성전자',
    years=[2022, 2023],
    reprt_codes=[ReportCode.ANNUAL],
    sections=[SectionCode.DIVIDEND, SectionCode.EMPLOYEE],
)
for p in paths:
    print(p)

## 8. 캐시 갱신

In [ ]:
reader.refresh_corp_cache()
print('캐시 갱신 완료')

---
# 실험: 코스피·코스닥 증권사 재무건전성 일괄 수집

**목표**: 상장 증권사 전체의 사업보고서에서 `재무건전성` 섹션을 HTML 뷰어로 조회해 `.md` 파일로 저장한다.

**흐름**
1. 전체 기업 코드 캐시에서 종목코드가 있는 상장사 필터링
2. DART 기업개황 API로 업종(증권) 확인
3. 각 증권사의 최신 사업보고서 접수번호 조회
4. `fetch_document(section_filter=['재무건전성'])` 으로 뷰어 HTML 수집
5. 배치 ID 폴더에 `.md` 저장


### Step 1. 증권사 기업 목록 구성

In [ ]:
import requests, time, json
from pathlib import Path

API_KEY = os.environ['DART_API_KEY']
BASE    = 'https://opendart.fss.or.kr/api'

def dart_get(endpoint, params):
    """DART API GET 요청 공통 함수."""
    params['crtfc_key'] = API_KEY
    url = f"{BASE}/{endpoint}.json?" + "&".join(f"{k}={v}" for k, v in params.items())
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

# 전체 캐시에서 상장사(종목코드 있는 기업)만 추출
import json as _json
cache_path = Path('/content/dart_cache/corp_codes.json')
all_corps  = _json.loads(cache_path.read_text(encoding='utf-8'))
listed     = [c for c in all_corps if c.get('stock_code', '').strip()]
print(f'전체 상장사: {len(listed)}개')

In [ ]:
# DART 기업개황 API로 업종 확인해 증권사 필터링
# 증권업 표준산업분류: K66 (금융 및 보험 관련 서비스업) 중 증권 관련
# 실무적으로는 기업명에 '증권'이 포함된 상장사를 우선 필터링 후 개황 확인

# 1차: 이름 기반 후보 추출 (빠름)
name_candidates = [
    c for c in listed
    if '증권' in c['corp_name']
]
print(f'이름 기반 증권사 후보: {len(name_candidates)}개')
for c in name_candidates:
    print(f"  {c['corp_name']:20s}  {c['stock_code']}")

In [ ]:
# 2차: DART 기업개황 API로 업종 코드 검증
# ind_tp(업종): J=금융 관련, 더 구체적으로 업종_nm 필드 확인

securities_corps = []
INDUSTRY_KEYWORDS = ['증권', '선물', '자산운용']  # 포함 시 증권사로 분류

print('기업개황 조회 중...')
for corp in name_candidates:
    try:
        data = dart_get('company', {'corp_code': corp['corp_code']})
        ind  = data.get('induty_code', '') or ''
        name = data.get('corp_name', corp['corp_name'])
        # 업종코드 K66: 금융 및 보험업 서비스, 또는 이름에 증권 포함
        if ind.startswith('K66') or any(kw in name for kw in INDUSTRY_KEYWORDS):
            securities_corps.append({
                'corp_code':  corp['corp_code'],
                'corp_name':  name,
                'stock_code': corp['stock_code'],
                'ind_code':   ind,
            })
        time.sleep(0.1)  # API 요청 제한 준수
    except Exception as e:
        print(f"  ⚠ {corp['corp_name']}: {e}")

print(f'\n확인된 증권사: {len(securities_corps)}개')
for c in securities_corps:
    print(f"  {c['corp_name']:20s}  {c['stock_code']}  ind={c['ind_code']}")

### Step 2. 각 증권사 최신 사업보고서 접수번호 조회

In [ ]:
TARGET_YEAR = 2024  # 조회할 사업연도

def get_rcept_no(corp_code: str, year: int) -> str:
    """해당 연도 사업보고서 접수번호 반환. 없으면 빈 문자열."""
    try:
        data = dart_get('list', {
            'corp_code':    corp_code,
            'bgn_de':       f'{year}0101',
            'end_de':       f'{year+1}0630',
            'pblntf_ty':    'A',
            'last_reprt_at':'Y',
        })
        for item in data.get('list', []):
            if '사업보고서' in item.get('report_nm', ''):
                return item['rcept_no']
    except Exception:
        pass
    return ''

print(f'{TARGET_YEAR}년 사업보고서 접수번호 조회 중...')
for corp in securities_corps:
    rcept_no = get_rcept_no(corp['corp_code'], TARGET_YEAR)
    corp['rcept_no'] = rcept_no
    status = rcept_no if rcept_no else '❌ 없음'
    print(f"  {corp['corp_name']:20s}  {status}")
    time.sleep(0.1)

# 접수번호 없는 기업 제외
target_corps = [c for c in securities_corps if c['rcept_no']]
print(f'\n처리 대상: {len(target_corps)}개 기업')

### Step 3. 재무건전성 섹션 수집 & MD 저장

In [ ]:
from dart_report_reader.vault.md_builder import new_batch_id
from dart_report_reader.parser.document_parser import DocumentParser, TocParser
import zipfile, re, io

SECTION_KEYWORDS = ['재무건전성']   # 조회할 섹션 키워드
BATCH_ID         = new_batch_id()   # 이번 실험 배치 ID
SLEEP_SEC        = 1.5              # 뷰어 요청 간 대기 (서버 부하 방지)

print(f'배치 ID: {BATCH_ID}')
print(f'저장 위치: /content/dart_vault/{BATCH_ID}/')
print(f'대상: {len(target_corps)}개 기업  /  섹션 키워드: {SECTION_KEYWORDS}\n')

results_log = []  # 결과 로그

for i, corp in enumerate(target_corps, 1):
    corp_name = corp['corp_name']
    rcept_no  = corp['rcept_no']
    print(f'[{i:02d}/{len(target_corps)}] {corp_name} ({rcept_no})')

    try:
        # --- 1. ZIP 다운로드 & 목차 추출 ---
        zip_info = reader._document.fetch_zip_info(rcept_no)
        xml_bytes = zip_info['main_xml']
        dcm_no    = zip_info['dcm_no']

        if not dcm_no:
            print(f'  ⚠ dcmNo 없음 — 건너뜀')
            results_log.append({'corp': corp_name, 'status': 'no_dcm_no'})
            continue

        # --- 2. 목차 파싱 (regex, XML 파싱 없음) ---
        doc = reader._doc_parser.parse_toc(xml_bytes, rcept_no)
        doc.dcm_no = dcm_no

        # --- 3. 재무건전성 섹션 탐색 ---
        target_secs = [
            s for s in doc.all_sections_flat()
            if any(kw in s.title for kw in SECTION_KEYWORDS)
        ]

        if not target_secs:
            print(f'  ℹ 재무건전성 섹션 없음 (섹션 {len(doc.all_sections_flat())}개 중 미발견)')
            results_log.append({'corp': corp_name, 'status': 'section_not_found',
                                 'total_sections': len(doc.all_sections_flat())})
            continue

        print(f'  ✅ 발견: {[s.title for s in target_secs]}')

        # --- 4. 뷰어 HTML로 섹션 내용 로딩 ---
        for sec in target_secs:
            try:
                sec.content_md = reader._doc_parser.fetch_section_content(
                    rcept_no=rcept_no,
                    dcm_no=dcm_no,
                    ele_id=sec.tocid,
                )
                print(f'     eleId={sec.tocid}  {sec.title[:40]}  ({len(sec.content_md):,} chars)')
            except Exception as e:
                print(f'     ⚠ 섹션 로딩 실패 eleId={sec.tocid}: {e}')
            time.sleep(SLEEP_SEC)

        # --- 5. MD 저장 ---
        saved_path = reader._builder.save_document(
            doc,
            batch_id=BATCH_ID,
            section_filter=SECTION_KEYWORDS,
        )
        print(f'  💾 저장: {saved_path.name}  ({saved_path.stat().st_size:,} bytes)')
        results_log.append({'corp': corp_name, 'status': 'ok',
                             'path': str(saved_path),
                             'sections': [s.title for s in target_secs]})

    except Exception as e:
        print(f'  ❌ 오류: {e}')
        results_log.append({'corp': corp_name, 'status': 'error', 'error': str(e)})

    time.sleep(SLEEP_SEC)

print(f'\n완료: {sum(1 for r in results_log if r["status"]=="ok")}/{len(target_corps)}개 성공')

### Step 4. 결과 확인 & 다운로드

In [ ]:
# 결과 요약
import pandas as pd

df_log = pd.DataFrame(results_log)
print('=== 결과 요약 ===')
print(df_log[['corp','status']].to_string(index=False))

# 생성된 MD 파일 목록
vault_dir = Path(f'/content/dart_vault/{BATCH_ID}')
md_files  = list(vault_dir.glob('*.md')) if vault_dir.exists() else []
print(f'\n생성된 MD 파일: {len(md_files)}개')
for f in md_files:
    print(f'  {f.name}  ({f.stat().st_size:,} bytes)')

In [ ]:
# 첫 번째 파일 미리보기
if md_files:
    print(md_files[0].read_text(encoding='utf-8')[:2000])

In [ ]:
# ZIP으로 묶어서 다운로드
import shutil
from google.colab import files

zip_path = f'/content/securities_financial_soundness_{BATCH_ID}.zip'
shutil.make_archive(
    zip_path.replace('.zip', ''), 'zip',
    '/content/dart_vault', BATCH_ID
)
files.download(zip_path)